# Simmilarity Mappings and Retrieval System

This notebook focuses on working with the extracted features to produce vector embedings as well as the system by which similarity mapping will be done.

In [1]:
from pathlib import Path
from collections import defaultdict

from sklearn.metrics.pairwise import cosine_similarity


import numpy as np
import pandas as pd
from tqdm import tqdm
import pretty_midi

import sys
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.config.paths import MSD_METADATA_DIR, PROCESSED_DATA_DIR, LMD_MATCHED_DIR, NOTEBOOK_DATASETS_DIR

## Load Dataset

In [2]:
features_df = pd.read_pickle(f"{NOTEBOOK_DATASETS_DIR}/all_features_limit_1000.pkl")

MemoryError: 

In [ ]:
features_df.head()

,note_count,mean_pitch,std_pitch,pitch_range,mean_duration,std_duration,mean_velocity,std_velocity,num_files,mean_instruments,...,segment_chroma,chord_progression,harmonic_rhythm,segment_variation,pitch_entropy,duration_entropy,ioi_entropy,track_id,artist,title
0,18928,61.048394,13.679155,69,0.307508,0.390621,91.760936,23.248512,4,13.750000,...,"[[0.22918454935602645, 0.0, 0.1107296137338105...","[0, 9, 9, 9, 9, 9, 0, 9, 9, 9, 0, 9, 7, 0, 0, 0]",0.466667,0.077834,2.632983,0.975266,0.720061,TRAAAGR128F425B14B,Cyndi Lauper,Into The Nightlife
1,20863,63.037387,11.449555,65,0.166876,0.223345,105.646743,15.864140,6,10.000000,...,"[[0.2593950504122279, 0.0, 0.0, 0.287809349220...","[3, 3, 5, 0, 0, 5, 5, 0, 5, 5, 2, 2, 2, 2, 7, 2]",0.533333,0.133852,2.390657,0.717437,0.412685,TRAAAZF12903CCCF6B,Matthew Wilder,Break My Stride
2,14141,58.134785,10.822386,72,0.838205,1.199980,82.439714,28.306069,5,11.000000,...,"[[0.13705583756321987, 0.0, 0.1793570219963124...","[7, 7, 7, 7, 2, 7, 7, 7, 7, 7, 9, 9, 5, 0, 0, 5]",0.400000,0.106609,2.328768,0.792585,0.765635,TRAABVM128F92CA9DC,Tesla,Caught In A Dream
3,16762,63.042656,10.898794,74,0.632178,0.750975,88.762320,24.536796,5,9.600000,...,"[[0.2153846153843787, 0.0021978021977997826, 0...","[0, 0, 7, 7, 9, 7, 7, 7, 7, 9, 7, 9, 4, 11, 11...",0.600000,0.091275,2.397167,1.416827,0.713183,TRAABXH128F42955D6,Brian Wilson,Keep An Eye On Summer (Album Version)
4,12360,55.062136,11.614828,62,0.561734,0.645350,88.016019,19.987963,3,12.666667,...,"[[0.005479452054779508, 0.09863013698603115, 0...","[2, 2, 2, 9, 9, 2, 2, 2, 4, 2, 2, 2, 9, 2, 11, 2]",0.533333,0.128428,2.555363,1.223556,0.525736,TRAACQE12903CC706C,Old Man River,Summer


## Produce Vector Embeddings

Deal with n-grams using a n-gram vocabulary


In [ ]:
from collections import Counter

global_vocab = Counter()

for ngrams in features_df["melody_ngrams"]:
    global_vocab.update(ngrams)

In [ ]:
TOP_NGRAMS = 500

ngram_vocab = {
    ngram: i
    for i, (ngram, _)
    in enumerate(global_vocab.most_common(TOP_NGRAMS))
}

In [ ]:
def vectorise_counter(counter, vocab):

    vec = np.zeros(len(vocab))
    
    total = sum(counter.values())

    if total == 0:
        return vec

    for item, count in counter.items():

        if item in vocab:
            vec[vocab[item]] = count / total

    return vec

In [ ]:
def build_embedding(row, ngram_vocab):

    parts = []

    parts.append(row["total_chroma"])
    parts.append(row["low_chroma"])
    parts.append(row["mid_chroma"])
    parts.append(row["high_chroma"])

    parts.append(row["ioi_hist"])
    parts.append(row["duration_hist"])

    parts.append(np.array([
        row["density"],
        row["drum_density"],
        row["bass_density"],
        row["melody_density"]
    ]))

    parts.append(np.array([
        row["pitch_entropy"],
        row["duration_entropy"],
        row["ioi_entropy"]
    ]))

    parts.append(
        vectorise_counter(
            row["melody_ngrams"],
            ngram_vocab
        )
    )

    return np.concatenate(parts)

In [ ]:
X = np.vstack([
    build_embedding(row, ngram_vocab)
    for _, row in features_df.iterrows()
])

from sklearn.preprocessing import StandardScaler

X = StandardScaler().fit_transform(X)

In [ ]:
metadata = features_df[
    [
        "track_id",
        "artist",
        "title",
    ]
].copy()

In [ ]:
feature_cols = [
    c for c in features_df.columns
    if c not in [
        "track_id",
        "artist",
        "title",
        "path"
    ]
]

## Implement FAISS

Check ready for FAISS

In [ ]:
X = X.astype(np.float32)

FAISS needs L2 normalisation

In [ ]:
import faiss

faiss.normalize_L2(X)

Building teh index, here with cosine similarity

In [ ]:
d = X.shape[1]

index = faiss.IndexFlatIP(d)  # inner product = cosine (after normalisation)
index.add(X)

### Query Function

In [ ]:
def get_knn(song_id, k=10):
    query_vec = X[song_id].reshape(1, -1)

    distances, indices = index.search(query_vec, k + 1)

    # remove self-match
    neigh_ids = indices[0][1:]
    neigh_sims = distances[0][1:]

    return list(zip(neigh_ids, neigh_sims))

In [ ]:
metadata = pd.DataFrame({
    "id": range(len(features_df)),
    "track": features_df["title"],
    "artist": features_df["artist"],
})

In [ ]:
def get_knn_with_metadata(song_id, k=10):
    neighbours = get_knn(song_id, k)

    results = []
    for idx, score in neighbours:
        row = metadata.iloc[idx]
        results.append({
            "id": int(idx),
            "track": row["track"],
            "artist": row["artist"],
            "score": float(score)
        })

    return results

Query by name function. Coudl expand to not need exact fit?

In [ ]:
def find_song_id(track_name):
    match = metadata[metadata["track"] == track_name]
    if len(match) == 0:
        raise ValueError("Song not found")
    return match.iloc[0]["id"]

In [ ]:
def query_by_track_name(track_name, k=10):
    song_id = find_song_id(track_name)
    return get_knn_with_metadata(song_id, k)

def query_by_artist_name(artist_name, k=10):
    matches = metadata[metadata["artist"] == artist_name]
    if len(matches) == 0:
        raise ValueError("Artist not found")
    
    results = []
    for _, row in matches.iterrows():
        song_id = row["id"]
        neighbours = get_knn_with_metadata(song_id, k)
        results.append({
            "track": row["track"],
            "artist": row["artist"],
            "neighbours": neighbours
        })
    
    return results

Helper function to check how songs cluster or rather how well teh embeddings work.

In [ ]:
def inspect_similarity(song_id, k=10):
    neighbours = get_knn(song_id, k)

    for n in neighbours:
        print(n["track"], n["artist"], n["score"])

## Test System

Initially we see if the system works by quering the neighbourhood of one of the songs observed in previous notebooks as being present in teh dataset.

In [ ]:
query_by_track_name("Break My Stride", k=5)

[{'id': 523,
  'track': 'Instrumental 2',
  'artist': 'Neva Dinova',
  'score': 0.3372921645641327},
 {'id': 631,
  'track': 'What A Wonderful World',
  'artist': 'Ray Quinn',
  'score': 0.308857798576355},
 {'id': 251,
  'track': 'Day Ditty',
  'artist': 'Shudder To Think',
  'score': 0.30746060609817505},
 {'id': 287,
  'track': 'Du Willst Immer Nur F...',
  'artist': 'Bangbros',
  'score': 0.2921904921531677},
 {'id': 332,
  'track': 'Cómo Estás_ Querida',
  'artist': 'Sergio Denis',
  'score': 0.2791801989078522}]

The relatively low score of the neighbourhood is a bit disconcerning. Whatsmore, on inspection these songs don't come across simmilar to the root case.

To check that these score are meaningful, 1000 random song pairs are picked with scores examined.

In [ ]:
scores = []

for _ in range(1000):
    i = np.random.randint(len(X))
    j = np.random.randint(len(X))

    score = cosine_similarity(
        X[i:i+1],
        X[j:j+1]
    )[0,0]

    scores.append(score)

print("Mean:", np.mean(scores))
print("Median:", np.median(scores))
print("Standard Deviation:", np.std(scores))
print("Minimum:", np.min(scores))
print("Maximum:", np.max(scores))

Mean: 0.004047985
Median: -0.014080951
Standard Deviation: 0.12623724
Minimum: -0.4281176
Maximum: 1.0000002


With median and mean close to 0 and a relatively low spread, the neighbourhood scores of around 0.3 do have some value to them. In that case, this may point to some features being misleading or perhaps some features should be weighted heavier.

Anotehr song is checked since one track cannot be informative over the whole system. Having said that though, two is not much stronger insight and so a more comprehensive method of testing needs to be established.

In [ ]:
metadata[ metadata["artist"].str.contains("Michael Jackson") ]

,id,track,artist
213,213,Man In The Mirror,Michael Jackson
393,393,Black Or White,Michael Jackson
944,944,Ben,Michael Jackson


In [ ]:
query_by_track_name("Man In The Mirror", k=5)

[{'id': 208,
  'track': 'Un Garcon Pas Comme Les Autres (Ziggy)',
  'artist': 'Céline Dion',
  'score': 0.5737673044204712},
 {'id': 233,
  'track': 'Borealis',
  'artist': 'DJ Eco',
  'score': 0.5713085532188416},
 {'id': 788,
  'track': 'Building A Mystery (Live)',
  'artist': 'Sarah McLachlan',
  'score': 0.5587574243545532},
 {'id': 643,
  'track': 'Dude (Looks Like A Lady)',
  'artist': 'Aerosmith',
  'score': 0.537179172039032},
 {'id': 712,
  'track': 'Sweet Harmony',
  'artist': 'The Beloved',
  'score': 0.5328057408332825}]

The neighbourhood of scores here is higher. On inspection there is some overlap in feel but I again find them slightly dubious. It should be said that this notebook makes use of a truncated dataset only making use of the first 1000 songs meaning neighbourhoods could get stronger with the full dataset. However, it is seen that "Black Or White" from the same artist is also present and by feel is possible closer than "Dude (Looks Like A Lady)". This again highlights questions over tuning features but it should also be rememebered that some "feel" may be absent from the MIDI representations of these tracks compared to the version we are used to.

To check the entire space of scores, some visualisations are produced.